In [9]:
from unstructured.partition.pdf import partition_pdf

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

from langchain_core.documents import Document

In [11]:
elements = partition_pdf(
    filename="..\data\[GU] Calcul des besoins V12.pdf",
    strategy="auto",  # meilleur parsing --> faux hi-res est le meilleur avec tesseract
    #infer_table_structure=True,
)

No languages specified, defaulting to English.


In [ ]:
elements = partition_pdf(
    filename="..\data\[GU] Calcul des besoins V12.pdf",
    strategy="hi_res",  # meilleur parsing
)

In [12]:
for el in elements[:20]:
    print(el.category)

Title
Title
UncategorizedText
Footer
Title
Title
Title
Title
Title
UncategorizedText
Title
UncategorizedText
UncategorizedText
UncategorizedText
ListItem
ListItem
UncategorizedText
Title
NarrativeText
Footer


In [13]:
for el in elements[:10]:
    print(type(el))

<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Text'>
<class 'unstructured.documents.elements.Footer'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Title'>
<class 'unstructured.documents.elements.Text'>


In [14]:
el = elements[0]

print(el.text)
print(el.category)
print(el.metadata)

SAGE X3 – V12 – CALCUL DES BESOINS
Title


In [15]:
el.metadata.page_number
el.metadata.filename
el.metadata.coordinates
el.metadata.languages
el.metadata.last_modified

'2025-11-27T08:42:00'

In [16]:
for el in elements[:20]:

    print("=" * 50)

    print("TYPE :", type(el))
    print("CATEGORY :", el.category)

    print("\nTEXT:")
    print(el.text[:300])

    print("\nMETADATA:")
    print(el.metadata)

TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
SAGE X3 – V12 – CALCUL DES BESOINS

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
Guide Utilisateur

METADATA:
TYPE : <class 'unstructured.documents.elements.Text'>
CATEGORY : UncategorizedText

TEXT:
Version du 22/05/2024

METADATA:
TYPE : <class 'unstructured.documents.elements.Footer'>
CATEGORY : Footer

TEXT:
1

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
HISTORIQUE DU DOCUMENT

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
Date

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
BZ

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
Chapitre

METADATA:
TYPE : <class 'unstructured.documents.elements.Title'>
CATEGORY : Title

TEXT:
Descriptif

METADATA:
TYPE : <class 'unstructured.documents.elements.Te

'2'

In [22]:
markdown_text = ""

current_page = None

for el in elements:

    text = el.text.strip()

    if not text:
        continue

    page = getattr(el.metadata, "page_number", None)

    # Changement de page
    if page != current_page:

        markdown_text += f"\n\n<!-- PAGE: {page} -->\n\n"
        current_page = page

    # Tous les titres = H1
    if el.category == "Title":

        markdown_text += f"\n# {text}\n"

    else:
        markdown_text += f"\n{text}\n"

In [23]:
headers_to_split_on = [
    ("#", "header"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

header_docs = markdown_splitter.split_text(markdown_text)

In [24]:
import re

def extract_page(text):

    match = re.search(r"<!-- PAGE: (\d+) -->", text)

    if match:
        return int(match.group(1))

    return None

In [25]:
enhanced_docs = []

source_name = "[GU] Calcul des besoins V12.pdf"

for doc in header_docs:

    header = doc.metadata.get("header", "Sans titre")

    page = extract_page(doc.page_content)

    # Nettoyage du marqueur page
    clean_content = re.sub(
        r"<!-- PAGE: \d+ -->",
        "",
        doc.page_content
    ).strip()

    # IMPORTANT :
    # on injecte le titre dans le contenu
    final_content = f"""
Section: {header}

{clean_content}
"""

    enhanced_doc = Document(
        page_content=final_content,
        metadata={
            "header": header,
            "page": page,
            "source": source_name,
        }
    )

    enhanced_docs.append(enhanced_doc)

In [40]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)

final_docs = recursive_splitter.split_documents(
    enhanced_docs
)

In [41]:
Document(
    page_content="""
Section: Article A445

Lorem ipsum...
""",

    metadata={
        "header": "Article A445",
        "page": 12,
        "source": "[GU] Calcul des besoins V12.pdf"
    }
)

Document(metadata={'header': 'Article A445', 'page': 12, 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='\nSection: Article A445\n\nLorem ipsum...\n')

In [42]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [43]:
from dotenv import load_dotenv

In [30]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13196.00it/s]


In [44]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=final_docs,
    embedding=embedding_model,
    persist_directory="./chroma_db_lorem3"
)

In [45]:
vdb=Chroma(persist_directory="./chroma_db_lorem3",
        embedding_function=embedding_model)

In [53]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(final_docs)

bm25_retriever.k = 20

In [46]:
retriever=vdb.as_retriever(search_kwargs={"k": 30})

In [58]:
from langchain.retrievers.ensemble import EnsembleRetriever

ModuleNotFoundError: No module named 'langchain.retrievers'

In [64]:
from langchain.retrievers import PineconeHybridSearchRetriever

ModuleNotFoundError: No module named 'langchain.retrievers'

In [59]:
import langchain
print(langchain.__version__)

1.3.1


In [60]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tchouarr\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2939.27it/s]


In [61]:
def rerank_docs(question, docs, top_k=10):

    pairs = [
        (question, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))

    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )

    reranked_docs = [
        doc for doc, score in scored_docs[:top_k]
    ]

    return reranked_docs

In [47]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [48]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}
Si possible de donné le numero de la page,
Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [49]:
chain = prompt | llm | StrOutputParser()

In [50]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        formatted.append(f"""
Source: {doc.metadata.get("source")}
Page: {doc.metadata.get("page")}
Section: {doc.metadata.get("header")}

{doc.page_content}
""")

    return "\n\n".join(formatted)

In [51]:
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le besoin est calculé en suivant les étapes suivantes :

1. Calcul du seuil de réappro : 
Seuil = Consommation de la période explorée / nb jours de la période explorée x nb jours conso pour le calcul du seuil (fonction du type de délai).

2. Calcul du disponible théorique : 
Dispo = Stock physique - Alloué - Manquants + Encours fournisseurs - Encours services

3. Comparaison seuil de réappro par rapport au disponible théorique :
Si le disponible théorique est inférieur au seuil de réappro, alors le besoin est égal au seuil de réappro.

4. Calcul du besoin net :
Besoin net = Besoin brut - disponible théorique

Ces informations sont présentes dans les pages suivantes :
- Page 15 (Section : Si Seuil de réappro > Disponible théorique)
- Page None (Section : L’algorithme de calcul est le suivant)
- Page None (Section : → Seuil = Stock de sécurité)
- Page None (Section : Besoin net = Besoin brut - disponible théorique)


## This one using the re-ranker

In [62]:
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)
docs = rerank_docs(question, docs)
context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le besoin brut est calculé comme suit : 
Besoins brut = Consommation de la période explorée / nb jours de la période explorée x nb jours conso pour le calcul du besoin (fonction du type de délai).
Cette information se trouve à la page 15.

Le besoin net est calculé comme suit : 
Besoins net = Besoin brut - disponible théorique.
Cette information se trouve dans la section "Besoin net = Besoin brut - disponible théorique" sans numéro de page précis.
